# TrustOCT — Training Notebook 1: Baseline Models
### Models: `resnet50` + `msf_resnet50`              
---
Run this notebook on **Account 1**. After training completes, results are saved to `outputs/` and synced to shared Google Drive.

In [ ]:
import os

if not os.path.exists('src'):
    !git clone https://github.com/Gnanapravallika/Trusthworthy_OCTAI.git
    %cd Trusthworthy_OCTAI
else:
    print('Repository already cloned.')

try:
    import albumentations
    import ptflops
except ImportError:
    !pip install -r requirements.txt

In [ ]:
import os
import sys
import time
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models
from PIL import Image
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Running on device: {device}')

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Running locally. Skipping Google Drive mount.')

In [ ]:
if not os.path.exists('/content/Kermany') and not os.path.exists('/content/OCT2017'):
    try:
        from google.colab import files
        print("Please upload your kaggle.json API token file:")
        uploaded = files.upload()
        if 'kaggle.json' in uploaded:
            !mkdir -p ~/.kaggle
            !cp kaggle.json ~/.kaggle/
            !chmod 600 ~/.kaggle/kaggle.json
            print("Downloading Kermany OCT2017 dataset from Kaggle...")
            !kaggle datasets download -d paultimothymooney/kermany2018 --unzip -p /content/Kermany
            print("Dataset downloaded successfully.")
    except Exception as e:
        print(f"Download skipped: {e}")

In [ ]:
from src.datasets.dataset import auto_detect_columns, patient_level_split

drive_base = '/content/drive/MyDrive'
shared_folder = '/content/drive/MyDrive/TrustOCT_Results'
os.makedirs(shared_folder, exist_ok=True)
os.makedirs(os.path.join(shared_folder, 'outputs'), exist_ok=True)

csv_path = 'kermany_dataset_mapping.csv'

# Check shared folder first
if not os.path.exists(csv_path):
    shared_csv = os.path.join(shared_folder, 'kermany_dataset_mapping.csv')
    if os.path.exists(shared_csv):
        import shutil
        shutil.copy(shared_csv, csv_path)
        print(f'Copied CSV from shared folder.')

if not os.path.exists(csv_path):
    csv_path = os.path.join(drive_base, 'kermany_dataset_mapping.csv')

if not os.path.exists(csv_path):
    print('CSV mapping file missing. Generating dynamically from directories...')
    root_oct = '/content/Kermany/OCT2017 '
    if not os.path.exists(root_oct):
        root_oct = '/content/Kermany'
    if not os.path.exists(root_oct):
        root_oct = '/content/OCT2017'
    
    if os.path.exists(root_oct):
        records = []
        class_to_idx = {'cnv': 0, 'dme': 1, 'drusen': 2, 'normal': 3}
        for root, dirs, files_list in os.walk(root_oct):
            for f in files_list:
                if f.lower().endswith(('.jpg', '.png', '.jpeg')):
                    parent_dir = os.path.basename(root)
                    lbl = class_to_idx.get(parent_dir.lower(), -1)
                    if lbl != -1:
                        base = os.path.splitext(f)[0]
                        parts = base.split('-')
                        pt_id = '-'.join(parts[:2]) if len(parts) >= 2 else base
                        records.append({
                            'image_path': os.path.join(root, f),
                            'label': lbl,
                            'patient_id': pt_id
                        })
        df_new = pd.DataFrame(records)
        df_new = df_new[df_new['label'] != -1]
        csv_path = 'kermany_dataset_mapping.csv'
        df_new.to_csv(csv_path, index=False)
        # Save to shared folder for other accounts
        df_new.to_csv(os.path.join(shared_folder, 'kermany_dataset_mapping.csv'), index=False)
        print(f'Created mapping CSV with {len(df_new)} images. Saved to shared folder.')

if os.path.exists(csv_path):
    df = auto_detect_columns(pd.read_csv(csv_path))
    
    # Force local path translation
    is_colab = os.path.exists('/content')
    local_kermany = '/content/Kermany'
    local_oct2017 = '/content/OCT2017'
    
    if is_colab and (os.path.exists(local_kermany) or os.path.exists(local_oct2017)):
        print('Force-directing paths to local storage...')
        def force_local_path(path_str):
            p = path_str.replace('\\', '/').replace('//', '/')
            parts = p.split('/')
            for folder in ['train', 'val', 'test']:
                if folder in parts:
                    idx = parts.index(folder)
                    subpath = '/'.join(parts[idx:])
                    for candidate in [
                        os.path.join(local_kermany, subpath),
                        os.path.join(local_kermany, 'OCT2017', subpath),
                        os.path.join(local_kermany, 'OCT2017 ', subpath),
                        os.path.join(local_oct2017, subpath),
                        os.path.join(local_oct2017, 'OCT2017', subpath)
                    ]:
                        if os.path.exists(candidate):
                            return candidate
            return path_str
        df['image_path'] = df['image_path'].apply(force_local_path)
    
    sample_img_path = df.iloc[0]['image_path']
    print(f'Sample image path: {sample_img_path}')
    print(f'Exists: {os.path.exists(sample_img_path)}')
    
    train_df, val_df, test_df = patient_level_split(df)
    print(f'Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}')
else:
    print('ERROR: Dataset files not found!')

In [ ]:
epochs = 30
lr = 1e-4
batch_size = 32

### 1. Train `resnet50` (Baseline)

In [ ]:
from src.training.trainer import run_experiment
run_experiment('resnet50', train_df, val_df, test_df, epochs=epochs, lr=lr, batch_size=batch_size, device_name=str(device))

### 2. Train `msf_resnet50` (+ MultiScale)

In [ ]:
from src.training.trainer import run_experiment
run_experiment('msf_resnet50', train_df, val_df, test_df, epochs=epochs, lr=lr, batch_size=batch_size, device_name=str(device))

### Sync Results to Shared Drive

In [ ]:
import shutil
shared_folder = '/content/drive/MyDrive/TrustOCT_Results/outputs'
os.makedirs(shared_folder, exist_ok=True)

for model_name in ['resnet50', 'msf_resnet50']:
    src = f'outputs/{model_name}'
    dst = os.path.join(shared_folder, model_name)
    if os.path.exists(src):
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f'Synced {model_name} to shared Drive.')

print('\nAll results from Notebook 1 synced successfully!')